In [1]:
# === Imports pour le modèle de recommandation ===

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# scikit-learn pour le ML
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# Pour sauvegarder le modèle
import pickle

# Configuration
pd.set_option('display.max_columns', None)
DATA_CLEAN = '../data/clean/'

# Chargement du dataset enrichi (1000 films cultes)
df = pd.read_csv(DATA_CLEAN + 'imdb_1000_enriched.csv')

print(f"Dataset chargé : {df.shape[0]} films, {df.shape[1]} colonnes")
print(f"\nAperçu des colonnes utiles :")
print(df[['movie_title', 'title_year', 'imdb_score', 'genre_1', 'genre_2', 'genre_3', 'duration']].head())

Dataset chargé : 1000 films, 51 colonnes

Aperçu des colonnes utiles :
                 movie_title  title_year  imdb_score genre_1    genre_2  \
0  The Shawshank Redemption         1994         9.3   Crime      Drama   
1           The Dark Knight         2008         9.0  Action      Crime   
2                 Inception         2010         8.8  Action  Adventure   
3                Fight Club         1999         8.8   Drama        NaN   
4              Pulp Fiction         1994         8.9   Crime      Drama   

  genre_3  duration  
0     NaN       142  
1   Drama       152  
2  Sci-Fi       148  
3     NaN       151  
4     NaN       178  


In [2]:
# === Encodage des genres en colonnes binaires (one-hot) ===

# Liste de tous les genres uniques dans nos 3 colonnes
tous_genres = pd.concat([df['genre_1'], df['genre_2'], df['genre_3']]).dropna().unique()
print(f"Nombre de genres distincts : {len(tous_genres)}")
print(f"Liste : {sorted(tous_genres)}")

# Pour chaque genre, on crée une colonne binaire (1 si le film a ce genre, 0 sinon)
for genre in tous_genres:
    df[f'is_{genre}'] = (
        (df['genre_1'] == genre) | 
        (df['genre_2'] == genre) | 
        (df['genre_3'] == genre)
    ).astype(int)

# Vérification
genres_cols = [col for col in df.columns if col.startswith('is_')]
print(f"\n{len(genres_cols)} colonnes binaires créées")
print(f"\nExemple sur Inception :")
inception_genres = df[df['movie_title'].str.strip() == 'Inception'][genres_cols].iloc[0]
print(inception_genres[inception_genres == 1])

Nombre de genres distincts : 21
Liste : ['Action', 'Adventure', 'Animation', 'Biography', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'History', 'Horror', 'Music', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Sport', 'Thriller', 'War', 'Western']

21 colonnes binaires créées

Exemple sur Inception :
is_Action       1
is_Adventure    1
is_Sci-Fi       1
Name: 2, dtype: int64


In [3]:
# === Sélection des features pour le modèle ===

# Features numériques pertinentes (basé sur l'EDA de la semaine 4)
features_num = ['imdb_score', 'duration', 'title_year', 'num_voted_users']

# Features catégorielles encodées (les genres)
features_cat = genres_cols  # toutes les colonnes is_*

# Liste finale
features = features_num + features_cat

print(f"Features numériques ({len(features_num)}) : {features_num}")
print(f"Features catégorielles ({len(features_cat)}) : {len(features_cat)} genres encodés")
print(f"Total features : {len(features)}")

# On crée une matrice X qui contient uniquement les features
X = df[features].copy()
print(f"\nMatrice X : {X.shape}")
print(f"Valeurs manquantes : {X.isnull().sum().sum()}")

Features numériques (4) : ['imdb_score', 'duration', 'title_year', 'num_voted_users']
Features catégorielles (21) : 21 genres encodés
Total features : 25

Matrice X : (1000, 25)
Valeurs manquantes : 0


In [4]:
# === Gestion des valeurs manquantes ===

# On remplit les valeurs manquantes par la médiane (plus robuste que la moyenne)
for col in features_num:
    X[col] = X[col].fillna(X[col].median())

print(f"Valeurs manquantes après remplissage : {X.isnull().sum().sum()}")
print(f"\nAperçu de X :")
X.head()

Valeurs manquantes après remplissage : 0

Aperçu de X :


,imdb_score,duration,title_year,num_voted_users,is_Crime,is_Action,is_Drama,is_Comedy,is_Adventure,is_Biography,is_Mystery,is_Horror,is_Western,is_Animation,is_Sci-Fi,is_Fantasy,is_Family,is_Romance,is_Musical,is_Thriller,is_Sport,is_War,is_Music,is_History,is_Documentary
0,9.3,142,1994,1689764,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,9.0,152,2008,1676169,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,8.8,148,2010,1468200,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
3,8.8,151,1999,1347461,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,8.9,178,1994,1324680,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [5]:
# === Normalisation des features ===

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Vérification
print(f"X_scaled : {X_scaled.shape}")
print(f"Moyennes (devraient être ≈ 0) : {X_scaled.mean(axis=0)[:4].round(2)}")
print(f"Écarts-types (devraient être ≈ 1) : {X_scaled.std(axis=0)[:4].round(2)}")

X_scaled : (1000, 25)
Moyennes (devraient être ≈ 0) : [ 0. -0. -0. -0.]
Écarts-types (devraient être ≈ 1) : [1. 1. 1. 1.]


In [6]:
# === Construction du modèle Nearest Neighbors ===

# n_neighbors = 6 car le 1er voisin sera toujours le film lui-même
# On veut donc 5 vrais voisins + 1 = 6
modele = NearestNeighbors(n_neighbors=6, metric='euclidean', algorithm='auto')

# Entraînement du modèle (= apprentissage de la position des 1000 films dans l'espace)
modele.fit(X_scaled)

print("✅ Modèle entraîné avec succès !")
print(f"Nombre de films dans le modèle : {modele.n_samples_fit_}")
print(f"Nombre de features par film : {modele.n_features_in_}")
print(f"Métrique de distance : {modele.metric}")

✅ Modèle entraîné avec succès !
Nombre de films dans le modèle : 1000
Nombre de features par film : 25
Métrique de distance : euclidean


In [7]:
# === Premier test : recommandations pour Inception ===

# On récupère l'index d'Inception dans le dataset
idx_inception = df[df['movie_title'].str.strip() == 'Inception'].index[0]
print(f"Inception est à l'index : {idx_inception}")

# On demande au modèle les 6 plus proches voisins
distances, indices = modele.kneighbors([X_scaled[idx_inception]])

print(f"\nDistances : {distances[0].round(2)}")
print(f"Indices : {indices[0]}")

# On affiche les films correspondants (en sautant le 1er = Inception lui-même)
print(f"\n🎬 Films recommandés pour 'Inception' :\n")
for i, (idx, dist) in enumerate(zip(indices[0][1:], distances[0][1:])):
    titre = df.iloc[idx]['movie_title'].strip()
    annee = df.iloc[idx]['title_year']
    note = df.iloc[idx]['imdb_score']
    genres = [df.iloc[idx][g] for g in ['genre_1', 'genre_2', 'genre_3'] if pd.notna(df.iloc[idx][g])]
    print(f"  {i+1}. {titre} ({annee}) - Note {note} - {', '.join(genres)}")
    print(f"     Distance : {dist:.2f}\n")

Inception est à l'index : 2

Distances : [0.   2.69 2.7  3.79 3.83 4.05]
Indices : [ 2 13  7  6 15 11]

🎬 Films recommandés pour 'Inception' :

  1. The Avengers (2012) - Note 8.1 - Action, Adventure, Sci-Fi
     Distance : 2.69

  2. The Matrix (1999) - Note 8.7 - Action, Sci-Fi
     Distance : 2.70

  3. The Lord of the Rings: The Fellowship of the Ring (2001) - Note 8.8 - Action, Adventure, Drama
     Distance : 3.79

  4. Batman Begins (2005) - Note 8.3 - Action, Adventure
     Distance : 3.83

  5. The Lord of the Rings: The Two Towers (2002) - Note 8.7 - Action, Adventure, Drama
     Distance : 4.05



In [8]:
# === Test sur plusieurs films pour valider la robustesse du modèle ===

def afficher_recommandations(titre_film):
    """Affiche les 5 films recommandés pour un film donné"""
    # Recherche du film (insensible à la casse, gestion des espaces)
    masque = df['movie_title'].str.strip().str.lower() == titre_film.lower().strip()
    
    if not masque.any():
        print(f"❌ Film '{titre_film}' non trouvé dans le dataset")
        return
    
    idx = df[masque].index[0]
    distances, indices = modele.kneighbors([X_scaled[idx]])
    
    # Infos du film de référence
    info = df.iloc[idx]
    print(f"🎬 Film de référence : {info['movie_title'].strip()} ({info['title_year']})")
    print(f"   Note : {info['imdb_score']} | Genres : {info['genre_1']}, {info['genre_2']}, {info['genre_3']}")
    print(f"\n   👉 5 recommandations :\n")
    
    for i, (idx_voisin, dist) in enumerate(zip(indices[0][1:], distances[0][1:])):
        v = df.iloc[idx_voisin]
        genres = [v[g] for g in ['genre_1', 'genre_2', 'genre_3'] if pd.notna(v[g])]
        print(f"   {i+1}. {v['movie_title'].strip()} ({v['title_year']}) - {v['imdb_score']}/10")
        print(f"      Genres : {', '.join(genres)} | Distance : {dist:.2f}")
    print("\n" + "="*70 + "\n")

# Test sur 3 films de genres très différents
afficher_recommandations("The Dark Knight")
afficher_recommandations("Forrest Gump")
afficher_recommandations("The Lion King")

🎬 Film de référence : The Dark Knight (2008)
   Note : 9.0 | Genres : Action, Crime, Drama

   👉 5 recommandations :

   1. The Shawshank Redemption (1994) - 9.3/10
      Genres : Crime, Drama | Distance : 2.48
   2. Pulp Fiction (1994) - 8.9/10
      Genres : Crime, Drama | Distance : 3.15
   3. Fight Club (1999) - 8.8/10
      Genres : Drama | Distance : 3.80
   4. The Lord of the Rings: The Fellowship of the Ring (2001) - 8.8/10
      Genres : Action, Adventure, Drama | Distance : 4.12
   5. The Lord of the Rings: The Return of the King (2003) - 8.9/10
      Genres : Action, Adventure, Drama | Distance : 4.36


🎬 Film de référence : Forrest Gump (1994)
   Note : 8.8 | Genres : Comedy, Drama, nan

   👉 5 recommandations :

   1. Fight Club (1999) - 8.8/10
      Genres : Drama | Distance : 2.34
   2. American Beauty (1999) - 8.4/10
      Genres : Drama | Distance : 3.24
   3. Pulp Fiction (1994) - 8.9/10
      Genres : Crime, Drama | Distance : 3.69
   4. The Lord of the Rings: The Fe

In [10]:
# === Fonction finale de recommandation ===

def recommander_films(titre_film, n=5):
    """
    Recommande n films similaires à un film donné.
    
    Paramètres
    ----------
    titre_film : str
        Le titre du film de référence
    n : int
        Nombre de recommandations souhaitées (défaut : 5)
    
    Retourne
    --------
    pd.DataFrame avec les colonnes : titre, année, note, genres, distance
    None si le film n'est pas trouvé
    """
    # Recherche du film
    masque = df['movie_title'].str.strip().str.lower() == titre_film.lower().strip()
    
    if not masque.any():
        print(f"❌ Film '{titre_film}' non trouvé dans le dataset")
        return None
    
    idx = df[masque].index[0]
    
    # Recherche des n+1 voisins (le 1er sera le film lui-même)
    distances, indices = modele.kneighbors([X_scaled[idx]], n_neighbors=n+1)
    
    # Construction du DataFrame de recommandations
    recos = []
    for idx_voisin, dist in zip(indices[0][1:], distances[0][1:]):
        v = df.iloc[idx_voisin]
        genres = ' | '.join([str(v[g]) for g in ['genre_1', 'genre_2', 'genre_3'] if pd.notna(v[g])])
        recos.append({
            'titre': v['movie_title'].strip(),
            'année': int(v['title_year']),
            'note': v['imdb_score'],
            'genres': genres,
            'distance': round(dist, 2)
        })
    
    return pd.DataFrame(recos)


# Test rapide de la fonction
print("Test de la fonction sur 'The Matrix' :\n")
recos = recommander_films("The Matrix", n=5)
print(recos)

Test de la fonction sur 'The Matrix' :

                        titre  année  note                       genres  \
0  Terminator 2: Judgment Day   1991   8.5              Action | Sci-Fi   
1                   Inception   2010   8.8  Action | Adventure | Sci-Fi   
2                The Avengers   2012   8.1  Action | Adventure | Sci-Fi   
3              The Terminator   1984   8.1              Action | Sci-Fi   
4                    Iron Man   2008   7.9  Action | Adventure | Sci-Fi   

   distance  
0      2.56  
1      2.70  
2      3.10  
3      3.60  
4      3.62  


In [11]:
# === Sauvegarde du modèle et des données associées ===

import os

# Création du dossier models s'il n'existe pas
os.makedirs('../models', exist_ok=True)

# On sauvegarde 4 éléments dans un seul fichier (un dictionnaire pickle)
elements_a_sauvegarder = {
    'modele': modele,
    'scaler': scaler,
    'X_scaled': X_scaled,
    'df': df,  # le dataframe avec tous les films
    'features': features  # liste des features utilisées
}

with open('../models/modele_reco.pkl', 'wb') as f:
    pickle.dump(elements_a_sauvegarder, f)

# Vérification
taille_mb = os.path.getsize('../models/modele_reco.pkl') / (1024 * 1024)
print(f"✅ Modèle sauvegardé : ../models/modele_reco.pkl")
print(f"   Taille : {taille_mb:.2f} MB")
print(f"\nContenu sauvegardé :")
for cle in elements_a_sauvegarder.keys():
    print(f"  • {cle}")

✅ Modèle sauvegardé : ../models/modele_reco.pkl
   Taille : 1.40 MB

Contenu sauvegardé :
  • modele
  • scaler
  • X_scaled
  • df
  • features
